In [2]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.optimizers import Adam

DATASET_DIR = "dataset/pac"   # unzip pac.zip manually

IMAGE_SIZE = (128, 128)

X = []
y = []

label_map = {"gen": 0, "fak": 1}  # 0 = genuine, 1 = fake

for folder in os.listdir(DATASET_DIR):
    folder_path = os.path.join(DATASET_DIR, folder)
    
    if not os.path.isdir(folder_path):
        continue

    print("Loading:", folder)

    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        img = cv2.imread(file_path)
        if img is None:
            print("Skipped:", file_path)
            continue

        img = cv2.resize(img, IMAGE_SIZE)
        img = img / 255.0

        X.append(img)

        # --- Label extraction ---
        if folder == "gen":
            y.append(0)
        elif folder == "fak":
            y.append(1)

X = np.array(X)
y = np.array(y)

print("Total images:", len(X))
print("Genuine count:", sum(y==0))
print("Fake count:", sum(y==1))

# Balance fix: Shuffle
indices = np.arange(len(X))
np.random.shuffle(indices)
X = X[indices]
y = y[indices]

# Train split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert labels to categorical
y_train_cat = to_categorical(y_train, 2)
y_test_cat = to_categorical(y_test, 2)

# -------------------------------
# MODEL
# -------------------------------
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(2, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(0.0003),
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=25,
    batch_size=16
)

model.save("package_verifier.h5")

print("Training complete!")


Loading: fak
Loading: gen
Total images: 16
Genuine count: 8
Fake count: 8


C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step - accuracy: 0.5833 - loss: 0.6928 - val_accuracy: 0.5000 - val_loss: 0.7174
Epoch 2/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 0.6914 - val_accuracy: 0.5000 - val_loss: 0.6767
Epoch 3/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1000ms/step - accuracy: 0.5833 - loss: 0.6221 - val_accuracy: 0.5000 - val_loss: 0.6641
Epoch 4/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 951ms/step - accuracy: 0.8333 - loss: 0.5820 - val_accuracy: 0.5000 - val_loss: 0.6665
Epoch 5/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.7500 - loss: 0.5821 - val_accuracy: 0.5000 - val_loss: 0.6644
Epoch 6/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.8333 - loss: 0.5214 - val_accuracy: 0.5000 - val_loss: 0.6759
Epoch 7/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.9167 - loss: 0.4797 - val_accuracy: 0.5000 - val_loss: 0.6871
Epoch 8/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.9167 - loss: 0.4295 - val_accuracy: 0.5000 - val_loss: 0.7027
Epoch 9

Training complete!
